In [1]:
# 1. Import thư viện xử lý dữ liệu
import pandas as pd
import numpy as np

# 2. Import công cụ làm việc với đường dẫn
from pathlib import Path

# 3. Xác định thư mục gốc của project
PROJECT_DIR = Path.cwd()

if not (PROJECT_DIR / "data" / "processed" / "train.csv").exists():
    PROJECT_DIR = PROJECT_DIR.parent

# 4. Xác định đường dẫn dữ liệu
DATA_DIR = PROJECT_DIR / "data" / "processed"

# 5. Đọc dữ liệu Train, Validation và Test
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "validation.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

# 6. Kiểm tra kích thước dữ liệu
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

# 7. Hiển thị 5 dòng đầu tiên
train_df.head()

Train: (315125, 5)
Validation: (67524, 5)
Test: (67524, 5)


,url,type,normalized_url,hostname,has_hostname
0,https://www.youtube.com,legitimate,https://www.youtube.com,www.youtube.com,True
1,https://www.facebook.com,legitimate,https://www.facebook.com,www.facebook.com,True
2,https://www.wikipedia.org,legitimate,https://www.wikipedia.org,www.wikipedia.org,True
3,https://www.reddit.com,legitimate,https://www.reddit.com,www.reddit.com,True
4,https://www.yahoo.com,legitimate,https://www.yahoo.com,www.yahoo.com,True


In [2]:
# Kiểm tra thông tin các cột
print("=== CẤU TRÚC TRAIN DATA ===")
train_df.info()

# Kiểm tra 5 dòng dữ liệu đầu tiên
print("\n=== 5 DÒNG DỮ LIỆU ĐẦU TIÊN ===")
display(train_df.head())

# Kiểm tra số lượng giá trị thiếu
print("\n=== GIÁ TRỊ THIẾU ===")
print(train_df.isnull().sum())

=== CẤU TRÚC TRAIN DATA ===
<class 'pandas.DataFrame'>
RangeIndex: 315125 entries, 0 to 315124
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   url             315125 non-null  str  
 1   type            315125 non-null  str  
 2   normalized_url  315125 non-null  str  
 3   hostname        315125 non-null  str  
 4   has_hostname    315125 non-null  bool 
dtypes: bool(1), str(4)
memory usage: 54.5 MB

=== 5 DÒNG DỮ LIỆU ĐẦU TIÊN ===


,url,type,normalized_url,hostname,has_hostname
0,https://www.youtube.com,legitimate,https://www.youtube.com,www.youtube.com,True
1,https://www.facebook.com,legitimate,https://www.facebook.com,www.facebook.com,True
2,https://www.wikipedia.org,legitimate,https://www.wikipedia.org,www.wikipedia.org,True
3,https://www.reddit.com,legitimate,https://www.reddit.com,www.reddit.com,True
4,https://www.yahoo.com,legitimate,https://www.yahoo.com,www.yahoo.com,True



=== GIÁ TRỊ THIẾU ===
url               0
type              0
normalized_url    0
hostname          0
has_hostname      0
dtype: int64


In [3]:
# Danh sách 10 đặc trưng sử dụng cho mô hình
FEATURE_COLUMNS = [
    "url_length",
    "hostname_length",
    "dot_count",
    "hyphen_count",
    "digit_count",
    "at_count",
    "slash_count",
    "question_count",
    "equal_count",
    "has_https"
]

# Nhãn cần dự đoán
TARGET_COLUMN = "type"

print("Số lượng đặc trưng:", len(FEATURE_COLUMNS))
print("Danh sách đặc trưng:")

for index, feature in enumerate(FEATURE_COLUMNS, start=1):
    print(f"{index}. {feature}")

Số lượng đặc trưng: 10
Danh sách đặc trưng:
1. url_length
2. hostname_length
3. dot_count
4. hyphen_count
5. digit_count
6. at_count
7. slash_count
8. question_count
9. equal_count
10. has_https


In [4]:
def extract_features(data):
    """
    Trích xuất 10 đặc trưng từ dữ liệu URL.

    Input:
        data: DataFrame chứa normalized_url và hostname

    Output:
        DataFrame chứa 10 đặc trưng
    """

    features = pd.DataFrame(index=data.index)

    # 1. Độ dài URL
    features["url_length"] = data["normalized_url"].str.len()

    # 2. Độ dài hostname
    features["hostname_length"] = data["hostname"].str.len()

    # 3. Số dấu chấm
    features["dot_count"] = data["normalized_url"].str.count(r"\.")

    # 4. Số dấu gạch ngang
    features["hyphen_count"] = data["normalized_url"].str.count("-")

    # 5. Số chữ số
    features["digit_count"] = data["normalized_url"].str.count(r"\d")

    # 6. Số ký tự @
    features["at_count"] = data["normalized_url"].str.count("@")

    # 7. Số dấu /
    features["slash_count"] = data["normalized_url"].str.count("/")

    # 8. Số dấu ?
    features["question_count"] = data["normalized_url"].str.count(r"\?")

    # 9. Số dấu =
    features["equal_count"] = data["normalized_url"].str.count("=")

    # 10. URL có sử dụng HTTPS hay không
    features["has_https"] = (
        data["normalized_url"]
        .str.lower()
        .str.startswith("https://")
        .astype(int)
    )

    return features

In [5]:
# Trích xuất đặc trưng từ dữ liệu Train
train_features = extract_features(train_df)

# Hiển thị 5 dòng đầu tiên
display(train_features.head())

# Kiểm tra kích thước dữ liệu
print("Kích thước dữ liệu:", train_features.shape)

# Kiểm tra danh sách cột
print("\nDanh sách đặc trưng:")
print(train_features.columns.tolist())

,url_length,hostname_length,dot_count,hyphen_count,digit_count,at_count,slash_count,question_count,equal_count,has_https
0,23,15,2,0,0,0,2,0,0,1
1,24,16,2,0,0,0,2,0,0,1
2,25,17,2,0,0,0,2,0,0,1
3,22,14,2,0,0,0,2,0,0,1
4,21,13,2,0,0,0,2,0,0,1


Kích thước dữ liệu: (315125, 10)

Danh sách đặc trưng:
['url_length', 'hostname_length', 'dot_count', 'hyphen_count', 'digit_count', 'at_count', 'slash_count', 'question_count', 'equal_count', 'has_https']


In [6]:
# 1. Kiểm tra kiểu dữ liệu của 10 đặc trưng
print("=== KIỂU DỮ LIỆU ===")
print(train_features.dtypes)

# 2. Kiểm tra giá trị thiếu
print("\n=== GIÁ TRỊ THIẾU ===")
print(train_features.isnull().sum())

# 3. Kiểm tra giá trị vô hạn
print("\n=== GIÁ TRỊ VÔ HẠN ===")
print(np.isinf(train_features.select_dtypes(include="number")).sum())

# 4. Thống kê tổng quan
print("\n=== THỐNG KÊ ĐẶC TRƯNG ===")
display(train_features.describe().round(2))

=== KIỂU DỮ LIỆU ===
url_length         int64
hostname_length    int64
dot_count          int64
hyphen_count       int64
digit_count        int64
at_count           int64
slash_count        int64
question_count     int64
equal_count        int64
has_https          int64
dtype: object

=== GIÁ TRỊ THIẾU ===
url_length         0
hostname_length    0
dot_count          0
hyphen_count       0
digit_count        0
at_count           0
slash_count        0
question_count     0
equal_count        0
has_https          0
dtype: int64

=== GIÁ TRỊ VÔ HẠN ===
url_length         0
hostname_length    0
dot_count          0
hyphen_count       0
digit_count        0
at_count           0
slash_count        0
question_count     0
equal_count        0
has_https          0
dtype: int64

=== THỐNG KÊ ĐẶC TRƯNG ===


,url_length,hostname_length,dot_count,hyphen_count,digit_count,at_count,slash_count,question_count,equal_count,has_https
count,315125.00,315125.00,315125.00,315125.00,315125.00,315125.00,315125.00,315125.00,315125.00,315125.00
mean,59.84,19.13,2.60,1.22,4.22,0.01,4.43,0.15,0.26,0.78
std,38.42,6.63,1.15,2.52,9.48,0.10,1.57,0.38,0.91,0.41
min,12.00,1.00,0.00,0.00,0.00,0.00,2.00,0.00,0.00,0.00
25%,40.00,15.00,2.00,0.00,0.00,0.00,3.00,0.00,0.00,1.00
50%,52.00,18.00,2.00,0.00,1.00,0.00,4.00,0.00,0.00,1.00
75%,70.00,22.00,3.00,1.00,6.00,0.00,5.00,0.00,0.00,1.00
max,2314.00,240.00,29.00,42.00,573.00,10.00,46.00,15.00,34.00,1.00


In [7]:
# Trích xuất đặc trưng cho Validation
val_features = extract_features(val_df)

# Trích xuất đặc trưng cho Test
test_features = extract_features(test_df)

# Kiểm tra kích thước
print("Train features:", train_features.shape)
print("Validation features:", val_features.shape)
print("Test features:", test_features.shape)

# Kiểm tra tên cột có đồng nhất không
print("\nCác tập có cùng danh sách đặc trưng:",
      list(train_features.columns) ==
      list(val_features.columns) ==
      list(test_features.columns))

Train features: (315125, 10)
Validation features: (67524, 10)
Test features: (67524, 10)

Các tập có cùng danh sách đặc trưng: True


In [8]:
# Tách dữ liệu Train
X_train = train_features[FEATURE_COLUMNS]
y_train = train_df[TARGET_COLUMN]

# Tách dữ liệu Validation
X_val = val_features[FEATURE_COLUMNS]
y_val = val_df[TARGET_COLUMN]

# Tách dữ liệu Test
X_test = test_features[FEATURE_COLUMNS]
y_test = test_df[TARGET_COLUMN]

# Kiểm tra kích thước
print("=== KÍCH THƯỚC DỮ LIỆU ===")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

# Kiểm tra phân bố nhãn
print("\n=== PHÂN BỐ NHÃN TRAIN ===")
print(y_train.value_counts())

=== KÍCH THƯỚC DỮ LIỆU ===
X_train: (315125, 10)
y_train: (315125,)
X_val: (67524, 10)
y_val: (67524,)
X_test: (67524, 10)
y_test: (67524,)

=== PHÂN BỐ NHÃN TRAIN ===
type
legitimate    242019
phishing       73106
Name: count, dtype: int64


In [9]:
# Kiểm tra phân bố nhãn của cả 3 tập dữ liệu
for name, labels in [
    ("Train", y_train),
    ("Validation", y_val),
    ("Test", y_test)
]:
    print(f"\n=== {name} ===")
    print(labels.value_counts())
    print("\nTỷ lệ phần trăm:")
    print((labels.value_counts(normalize=True) * 100).round(2))


=== Train ===
type
legitimate    242019
phishing       73106
Name: count, dtype: int64

Tỷ lệ phần trăm:
type
legitimate    76.8
phishing      23.2
Name: proportion, dtype: float64

=== Validation ===
type
legitimate    51861
phishing      15663
Name: count, dtype: int64

Tỷ lệ phần trăm:
type
legitimate    76.8
phishing      23.2
Name: proportion, dtype: float64

=== Test ===
type
legitimate    51858
phishing      15666
Name: count, dtype: int64

Tỷ lệ phần trăm:
type
legitimate    76.8
phishing      23.2
Name: proportion, dtype: float64
